# Notebook 02d — Pseudonymisation Pipeline

**Purpose.** Replace Reddit usernames with deterministic salted-hash pseudonyms across all working files, and scrub Reddit username mentions from comment body text. This addresses the GDPR/AoIR ethics requirement that personally identifiable information (PII) should not be retained in research working files once it is no longer needed for analysis.

**Position in the pipeline.** This notebook should run **once**, after LLM scoring (02c) and before all downstream analysis (03, 04). It is a one-shot data-handling step, not part of the analytical workflow.

**Inputs.**
- `comments_raw_full.csv` — raw scrape with real Reddit usernames in the `author` column.
- `comments_scored_llm.csv` — LLM-scored corpus (no author column at the moment; only `body` text needs scrubbing).
- Optional: `validation_*.csv`, `author_lca_classes.csv` — any downstream file that may carry author info.

**Outputs.**
- `comments_scored_llm_anon.csv` — scored corpus with `author` pseudonym column merged in and `/u/...` mentions scrubbed from `body`.
- `comments_raw_anon.csv` — raw scrape with `author` replaced by pseudonym.
- `pseudonym_mapping.csv` — `{author → author_pseudo}` lookup table (treat as restricted; keep outside any shared/version-controlled directory).
- Anonymised versions of any other identified file.

**What this notebook does NOT do.** It does not re-run LLM scoring. The LLM never saw the `author` column (only `body` text); the construct scores in `comments_scored_llm.csv` are unaffected by pseudonymisation. Notebook 04's analyses re-run on the pseudonymised file produce byte-identical results because the pseudonym is deterministic per real user — same number of unique authors, same comments-per-author counts, same crossing structure between authors and influencers.

**Ethics framework references.** Association of Internet Researchers (Franzke et al., 2020); British Psychological Society Code of Human Research Ethics (2021); UK Data Protection Act 2018 (research exemption, Schedule 2).

## 0. Setup

In [ ]:
import os
import re
import hashlib
import getpass
import pandas as pd
from pathlib import Path

BASE = Path.cwd()
print(f'Working directory: {BASE}')

## 1. Salt configuration

**Why a salt?** A plain SHA-256 hash of a Reddit username is reversible by anyone who can guess or brute-force the input — Reddit usernames are short and the namespace is enumerable. Adding a secret salt before hashing makes the pseudonym one-way in practice: an adversary who obtains the pseudonym CSV without the salt cannot recover the original usernames.

**How to set the salt.**

Option A (recommended). Create a file called `.env` in the working directory containing one line: `PSEUDONYM_SALT=<random_string>`. Add `.env` to your `.gitignore` if you use git. Pick a random string of at least 32 characters — e.g., the output of `python -c 'import secrets; print(secrets.token_hex(32))'`.

Option B. Set the environment variable `PSEUDONYM_SALT` in your shell before launching Jupyter.

Option C. Type it in interactively (the cell below will prompt). Useful for a one-off run but easy to lose; write the salt down somewhere secure if you go this route — you need the SAME salt every time you want the SAME pseudonyms.

**Critical:** keep the salt secret and stable. If the salt changes, the pseudonyms change, and you cannot match a re-run to previous outputs.

In [ ]:
# Load the salt from .env, then from env var, then prompt as fallback.
SALT = None
env_path = BASE / '.env'
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if line.startswith('PSEUDONYM_SALT='):
            SALT = line.split('=', 1)[1].strip().strip('"').strip("'")
            print('Loaded salt from .env')
            break

if SALT is None:
    SALT = os.environ.get('PSEUDONYM_SALT')
    if SALT:
        print('Loaded salt from environment variable')

if SALT is None:
    print('No salt found in .env or environment.')
    SALT = getpass.getpass('Enter pseudonym salt (input will not echo): ')

assert SALT and len(SALT) >= 16, (
    'Salt is missing or too short. Use at least 16 characters; 32+ recommended.'
)
print(f'Salt loaded (length = {len(SALT)} chars). Salt itself is NOT printed.')

## 2. Generate the pseudonym mapping

**Procedure.** For each unique username in `comments_raw_full.csv`, compute SHA-256(salt + username), take the first 10 hexadecimal characters, and prefix with `auth_`. The truncation gives a 10-hex-character identifier (40-bit space, ~1.1 trillion values) — more than enough headroom for ~3,000 unique authors with no realistic collision risk, and short enough to be readable.

**Special handling of `[deleted]`.** Reddit returns the literal string `[deleted]` as the author when the original user has deleted their account or the moderators have removed the comment. Many distinct real users get collapsed into this one label. We map all of them to a single bucket `anon_deleted` rather than giving them a hash — they cannot be treated as a single author for cross-classified analysis, so author-level analyses should drop them (Notebook 04 already does this).

In [ ]:
def make_pseudonym(name, salt=SALT, n_hex=10):
    """Deterministic salted SHA-256 pseudonym; first n_hex hex chars."""
    if pd.isna(name) or str(name).strip() == '' or name == '[deleted]':
        return 'anon_deleted'
    h = hashlib.sha256((salt + str(name)).encode('utf-8')).hexdigest()
    return f'auth_{h[:n_hex]}'

# Read the raw scrape and build the mapping
raw_path = BASE / 'comments_raw_full.csv'
assert raw_path.exists(), f'Cannot find {raw_path}'

raw = pd.read_csv(raw_path)
assert 'author' in raw.columns, 'comments_raw_full.csv has no author column'
print(f'Raw scrape: {len(raw):,} rows; '
      f'{raw["author"].nunique():,} unique author strings '
      f'(including "[deleted]")')

# One mapping per (unique) username
unique_authors = raw['author'].dropna().unique()
mapping = pd.DataFrame({
    'author':        unique_authors,
    'author_pseudo': [make_pseudonym(a) for a in unique_authors],
})

# Sanity: no collisions among non-anon_deleted pseudonyms
real = mapping.query("author_pseudo != 'anon_deleted'")
n_real_users = real['author'].nunique()
n_unique_pseudos = real['author_pseudo'].nunique()
assert n_real_users == n_unique_pseudos, (
    f'COLLISION: {n_real_users} real users → {n_unique_pseudos} pseudonyms. '
    'Increase n_hex in make_pseudonym().'
)
print(f'Mapping built: {n_real_users:,} real users → '
      f'{n_unique_pseudos:,} pseudonyms, 0 collisions.')

# Save the mapping. Treat this CSV as RESTRICTED — do not share or commit.
mapping_path = BASE / 'pseudonym_mapping.csv'
mapping.to_csv(mapping_path, index=False)
print(f'Saved mapping → {mapping_path.name} '
      '(RESTRICTED: keep outside any shared/version-controlled location)')

## 3. Apply pseudonyms to the LLM-scored corpus

Two operations on `comments_scored_llm.csv`:

1. **Merge in the `author` pseudonym** keyed by comment `id`. After this step, downstream notebooks (03, 04) can read `author` directly from the scored file and skip the raw-scrape merge entirely. The column is named `author` (not `author_pseudo`) so existing analysis code in Notebook 04 works unchanged.
2. **Scrub Reddit-style mentions inside `body` text** — `/u/username`, `u/username`, and capitalised variants. These are PII embedded in comment content and should not persist in working files. The mentions are replaced with `/u/redacted`. (The LLM has already scored the body text; the construct scores are unaffected.)

In [ ]:
scored_path = BASE / 'comments_scored_llm.csv'
assert scored_path.exists(), f'Cannot find {scored_path}'

scored = pd.read_csv(scored_path)
print(f'Scored corpus: {len(scored):,} rows, '
      f'{len(scored.columns)} columns')

# --- 3.1 Merge in pseudonym keyed by comment id ---
# Build id -> author_pseudo lookup
id_to_pseudo = raw[['id', 'author']].copy()
id_to_pseudo['author_pseudo'] = id_to_pseudo['author'].apply(make_pseudonym)
id_to_pseudo = id_to_pseudo[['id', 'author_pseudo']]
id_to_pseudo['id'] = id_to_pseudo['id'].astype(str)
scored['id']        = scored['id'].astype(str)

# Drop any pre-existing author column (in case of a previous mistake)
if 'author' in scored.columns:
    print('Dropping pre-existing author column from scored corpus.')
    scored = scored.drop(columns=['author'])

scored = scored.merge(id_to_pseudo, on='id', how='left')
scored = scored.rename(columns={'author_pseudo': 'author'})

n_matched   = scored['author'].notna().sum()
n_unmatched = scored['author'].isna().sum()
print(f'Pseudonym merge: {n_matched:,}/{len(scored):,} rows matched, '
      f'{n_unmatched:,} unmatched')

# --- 3.2 Scrub /u/username mentions in body ---
# Reddit usernames are 3-20 chars, [A-Za-z0-9_-]. Match with optional
# leading slash and case-insensitive 'u'. Avoid matching mid-URL by
# requiring a non-word boundary on the left.
MENTION_RE = re.compile(r'(?<!\w)/?[uU]/[A-Za-z0-9_-]{3,21}\b')

if 'body' in scored.columns:
    n_with_mention_before = scored['body'].str.contains(
        MENTION_RE, na=False, regex=True).sum()
    scored['body'] = scored['body'].astype(str).str.replace(
        MENTION_RE, '/u/redacted', regex=True)
    n_with_mention_after = scored['body'].str.contains(
        MENTION_RE, na=False, regex=True).sum()
    print(f'/u/mention scrub in body: {n_with_mention_before:,} rows had '
          f'mentions before, {n_with_mention_after:,} after (should be 0).')
else:
    print('No body column in scored corpus — nothing to scrub.')

# --- 3.3 Save with _anon suffix ---
scored_out = BASE / 'comments_scored_llm_anon.csv'
scored.to_csv(scored_out, index=False)
print(f'Saved → {scored_out.name}')

## 4. Anonymise the raw scrape

Produce an anonymised version of `comments_raw_full.csv` with `author` replaced by the pseudonym, and `/u/...` mentions scrubbed from `body`. This becomes the safe-to-keep version of the raw data. The original `comments_raw_full.csv` should then be encrypted at rest or deleted (see §6 below).

In [ ]:
raw_anon = raw.copy()
raw_anon['author'] = raw_anon['author'].apply(make_pseudonym)

if 'body' in raw_anon.columns:
    n_before = raw_anon['body'].astype(str).str.contains(
        MENTION_RE, na=False, regex=True).sum()
    raw_anon['body'] = raw_anon['body'].astype(str).str.replace(
        MENTION_RE, '/u/redacted', regex=True)
    n_after = raw_anon['body'].astype(str).str.contains(
        MENTION_RE, na=False, regex=True).sum()
    print(f'Raw body /u/ scrub: {n_before:,} -> {n_after:,} (should be 0).')

raw_anon_path = BASE / 'comments_raw_anon.csv'
raw_anon.to_csv(raw_anon_path, index=False)
print(f'Saved → {raw_anon_path.name}')

## 5. Anonymise any other working files that carry author info

Three other files in the workspace may have an `author` column. The cell below checks each one and writes an anonymised copy if so. Validation files are scanned for `body` mentions as well.

In [ ]:
DOWNSTREAM_FILES = [
    'author_lca_classes.csv',
    'validation_coding_sheet.csv',
    'validation_two_coders.csv',
    'validation_user_coded.csv',
    'comments_raw.csv',  # the smaller raw scrape if present
]

for fname in DOWNSTREAM_FILES:
    fpath = BASE / fname
    if not fpath.exists():
        print(f'  [skip] {fname} not present')
        continue
    try:
        df_ = pd.read_csv(fpath)
    except Exception as e:
        print(f'  [skip] {fname} could not be read: {type(e).__name__}: {e}')
        continue

    changed = False
    if 'author' in df_.columns:
        df_['author'] = df_['author'].apply(make_pseudonym)
        changed = True
    if 'body' in df_.columns:
        df_['body'] = df_['body'].astype(str).str.replace(
            MENTION_RE, '/u/redacted', regex=True)
        changed = True
    if changed:
        out = fpath.with_name(fpath.stem + '_anon.csv')
        df_.to_csv(out, index=False)
        print(f'  [ok] {fname:<35s} -> {out.name}')
    else:
        print(f'  [pass] {fname} has no author or body column to anonymise')

## 6. Verification — no real usernames remain in anonymised outputs

Sanity check: load each `_anon.csv` file, look for (a) any value in the `author` column that doesn't start with `auth_` or `anon_deleted`, and (b) any surviving `/u/<username>` patterns in any text column. Report either as a failure to fix before proceeding.

In [ ]:
ANON_OUTPUTS = list(BASE.glob('*_anon.csv'))
print(f'Found {len(ANON_OUTPUTS)} _anon.csv files to verify:')

all_clean = True
for fpath in ANON_OUTPUTS:
    try:
        df_ = pd.read_csv(fpath)
    except Exception as e:
        print(f'  [skip] {fpath.name}: {type(e).__name__}: {e}')
        continue

    issues = []
    # (a) Author column should only contain pseudonyms
    if 'author' in df_.columns:
        bad = df_[
            df_['author'].notna() &
            ~df_['author'].astype(str).str.startswith(('auth_', 'anon_'))
        ]
        if len(bad):
            issues.append(f'{len(bad)} non-pseudonym author values')
    # (b) No /u/<real_username> in any text column
    for col in df_.columns:
        if df_[col].dtype != object:
            continue
        # Skip pseudonym column itself
        if col == 'author':
            continue
        # Look for surviving mentions other than the redacted placeholder
        surviving = df_[col].astype(str).str.contains(
            r'(?<!\w)/?[uU]/(?!redacted\b)[A-Za-z0-9_-]{3,21}\b',
            na=False, regex=True
        )
        if surviving.any():
            issues.append(f"'{col}' has {surviving.sum()} surviving /u/ mentions")

    if issues:
        all_clean = False
        print(f'  [FAIL] {fpath.name}: ' + '; '.join(issues))
    else:
        print(f'  [ok]   {fpath.name}')

print()
if all_clean:
    print('All anonymised outputs are clean. Safe to promote to canonical filenames.')
else:
    print('One or more files still contain PII. Investigate before proceeding.')

## 7. Manual steps to complete the pipeline

Once the verification cell above prints `All anonymised outputs are clean`, the following manual steps promote the anonymised files to the canonical pipeline.

**Step 1 — Back up the originals to a secure location.** Move `comments_raw_full.csv`, `comments_raw.csv`, and any other file that still contains real usernames into a separate folder OUTSIDE the project working directory. Recommended practice: encrypt this folder with a password-protected archive (e.g., 7-Zip with AES-256, or macOS encrypted Disk Image) and label it as restricted. Retain for the minimum period required by the institutional ethics approval (typically 3–5 years), then destroy.

**Step 2 — Promote the `_anon` files to canonical names.** In a separate file-manager session (not this notebook, to keep the audit trail clean), rename:

- `comments_scored_llm_anon.csv` → `comments_scored_llm.csv` (overwrite)
- `comments_raw_anon.csv` → `comments_raw_full.csv` (overwrite)
- For each `<file>_anon.csv` produced in §5: `<file>_anon.csv` → `<file>.csv` (overwrite)

**Step 3 — Notebook 04 data-load simplification.** The current data-load cell in Notebook 04 merges the author column from `comments_raw_full.csv` via `id`. Because `comments_scored_llm.csv` now already contains the (pseudonymised) `author` column, the merge step is no longer needed. Replace the relevant block with a simpler check:

```python
df = pd.read_csv('comments_scored_llm.csv')
assert 'author' in df.columns, (
    'author column is missing — run Notebook 02d first to pseudonymise '
    'and merge author into the scored corpus.'
)
```

All other cells in Notebook 04 work unchanged. The cross-classified RE and the LCA produce results identical to the previous run because the pseudonyms are deterministic per real user.

**Step 4 — Lock down the pseudonym mapping.** Move `pseudonym_mapping.csv` and the salt (the `.env` file) into the same secure folder as the original raw scrape. These artefacts are required to reproduce the pseudonymisation deterministically and must be retained for the institutional data-retention period, then destroyed.
